# Conversation Loop with Recommender System

In [ ]:
import pandas as pd
import numpy as np
import pickle
import ast
import json
import re
from pathlib import Path
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
import google.generativeai as genai
import os
from dotenv import load_dotenv

: 

## 1. Load necessary file

In [ ]:
# all data
all_recipes_df = pd.read_csv("../../data/all_recipes_final.csv")

print(f"Loaded dataset: {len(all_recipes_df)} recipes")
print(f"Columns: {all_recipes_df.columns.tolist()}")

In [ ]:
# Load Vietnamese SBERT model
model = SentenceTransformer('keepitreal/vietnamese-sbert')
print(f"   Embedding dimension: {model.get_sentence_embedding_dimension()}")

In [ ]:
# embeddings for recipes
with open("recipes_embeddings_list.pkl", "rb") as f:
    recipes_embeddings_list = pickle.load(f)
print(f"Loaded {len(recipes_embeddings_list)} recipe embeddings")

## 2. Config LLM and Default_state

In [ ]:
# LLM Configuration
load_dotenv()
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
genai.configure(api_key=GOOGLE_API_KEY)

MODEL = "models/gemini-2.5-flash-lite"
STATE_FILE = "state6.json"

# Default state structure
DEFAULT_STATE = {
    "hard_constraints": {
        "type_of_food": [],
        "ingredients": [],
    },
    "soft_constraints": {
        "cook_time": [],
        "num_of_people": [],
        "calories": [],
        "algeric": []
    },
    "recommended_items": [],
    "accepted_items": [],
    "rejected_items": []
}

In [ ]:
# State management functions
def load_state():
    """Load dialogue state from JSON file"""
    if Path(STATE_FILE).exists():
        return json.load(open(STATE_FILE, "r", encoding="utf-8"))
    else:
        json.dump(DEFAULT_STATE, open(STATE_FILE, "w", encoding="utf-8"), indent=4, ensure_ascii=False)
        return DEFAULT_STATE

def save_state(state):
    """Save dialogue state to JSON file"""
    json.dump(state, open(STATE_FILE, "w", encoding="utf-8"), indent=4, ensure_ascii=False)

### 2.2 Buffer function

In [ ]:
# Global buffer to store recommended recipes data (like langchain memory)
RECOMMENDED_RECIPES_BUFFER = []

def add_recipes_to_buffer(recipes_data):
    global RECOMMENDED_RECIPES_BUFFER
    RECOMMENDED_RECIPES_BUFFER = recipes_data  # Replace with new recommendations

def get_recipes_buffer_text():
    if not RECOMMENDED_RECIPES_BUFFER:
        return "Chưa có món nào được gợi ý."
    
    buffer_text = []
    for recipe in RECOMMENDED_RECIPES_BUFFER:
        recipe_info = f"""
                      Món: {recipe['title']}
                      - Loại: {recipe['type_of_food']}
                      - Thời gian: {recipe['cook_time']}
                      - Số người: {recipe['num_of_people']}
                      - Nguyên liệu: {recipe.get('ingredients', 'N/A')}
                      - Mô tả: {recipe.get('description', 'N/A')}
                      - Các bước nấu: {recipe.get('step', 'N/A')}
                      - Lưu ý: {recipe.get('note', 'N/A')}
                      - Link: {recipe['link']}
                      """

        buffer_text.append(recipe_info.strip())    
    return "\n\n".join(buffer_text)

## 3. Dialogue Functions (from 3_Prompt_and_test_dialogue)

### STEP 1: PROMPT-BASED INTENT CLASSIFICATION

In [ ]:
# STEP 1: PROMPT-BASED INTENT CLASSIFICATION

def classify_intent(user_utterance):
    prompt = f"""
            Classify the USER INTENT in a conversational recommender system.

            Possible intents (can be multiple):
            - "Provide Preference" - user states what they want (food type, ingredients, cooking time, servings, calories)
            - "Inquire" - user asks questions
            - "Accept Recommendation" - user accepts/likes a recommended dish
            - "Reject Recommendation" - user rejects/dislikes a recommended dish

            Examples (in Vietnamese):
            - "Tôi muốn tìm món Tết" → ["Provide Preference"]
            - "Món nào nấu nhanh cho 2 người" → ["Provide Preference"]
            - "Bạn gợi ý món gì?" → ["Inquire"]
            - "Món này hay đấy" → ["Accept Recommendation"]
            - "Không, tôi không thích món này" → ["Reject Recommendation"]
            - "Tôi thích món này" → ["Accept Recommendation"]
            - "Cho tôi món khác" → ["Reject Recommendation"]

            User says: "{user_utterance}"

            Return ONLY a JSON array of intent strings (e.g., ["Provide Preference"]).
            """
    
    resp = genai.GenerativeModel(MODEL).generate_content(prompt)

    # Khối này dùng để format lại đầu ra của LLM (điển hình của gemini)
    try:
        text = resp.text.strip()
        # Remove markdown code blocks if present
        if "```" in text:
            parts = text.split("```")
            if len(parts) >= 2:
                text = parts[1]
                if text.startswith("json"):
                    text = text[4:]
                text = text.strip()
        
        text = text.strip()
        return json.loads(text)
    except Exception as e:
        print(f"[Error] parsing intent JSON: {e}")
        print(f"Response: {resp.text}")
        return []

### STEP 2: PROMPT-BASED STATE UPDATE

In [ ]:
# STEP 2: PROMPT-BASED STATE UPDATE

def update_state(user_utterance, intents, state):
    # Handle "Provide Preference" intent
    if "Provide Preference" in intents:
        # Detect if user is providing complete info upfront
        type_of_food_empty = len(state["hard_constraints"]["type_of_food"]) == 0
        ingredients_empty = len(state["hard_constraints"]["ingredients"]) == 0

        if type_of_food_empty and ingredients_empty:
            all_hard_empty = True # cả 2 trường đang rỗng
        else:
            all_hard_empty = False  # 1 trường rỗng hoặc cả 2 trường đã có dữ liệu

        
        # Find which field we're currently asking about
        current_field = None
        for key in ["type_of_food", "ingredients"]:
            if len(state["hard_constraints"][key]) == 0:
                current_field = key
                break
        
        # Build context for LLM
        field_context = ""
        if current_field and not all_hard_empty:
            field_names = {
                "type_of_food": "TYPE OF FOOD (type_of_food)",
                "ingredients": "INGREDIENTS (ingredients)"
            }
            field_context = f"\n**IMPORTANT: Currently asking about {field_names[current_field]}. ONLY update this field, KEEP all others unchanged.**\n"
        elif all_hard_empty:
            field_context = f"\n**IMPORTANT: User is providing complete information upfront. Update ALL fields mentioned in their utterance.**\n"
        
        prompt = f"""
                You are updating a conversational dialogue state JSON based on user preferences.

                User says: "{user_utterance}"
                {field_context}
                Current JSON state:
                {json.dumps(state, indent=4, ensure_ascii=False)}

                Rules:
                - **If user provides complete info in one utterance**: Update ALL mentioned fields
                - **If asking step-by-step**: ONLY update the field being asked, KEEP others unchanged
                - **Analyze carefully**: Identify which fields the user mentioned

                **Field-specific rules:**

                - **type_of_food**: 
                  + Update if user mentions food type: "món kho", "món xào", "món luộc",...
                    → Example: "món kho" → type_of_food = ["món kho"]
                  + If NOT mentioned → KEEP as []

                - **ingredients**: 
                  + Update if user mentions ingredients: "thịt lợn", "hải sản",...
                    - Specific: "thịt lợn" → ingredients = ["thịt lợn"]
                  + If NOT mentioned → KEEP as []
                  
                - **cook_time**: IMPORTANT (soft_constraints)
                  + Update if user mentions time (including "don't care")
                    - Specific: "45 phút" → ["45 phút"]
                  + If NOT mentioned → KEEP as []
                  
                - **algeric**: IMPORTANT (soft_constraints)
                  + Update if user mentions allergies (including "no allergies")
                    - "không dị ứng", "không bị dị ứng" → algeric = ["none"]
                    - Specific: "dị ứng tôm" → algeric = ["tôm"]
                  + If NOT mentioned → KEEP as []

                - **num_of_people**: IMPORTANT (soft_constraints)
                  + Update if user mentions servings (including "don't care")
                    - "không cần quan tâm số người" → num_of_people = ["none"]
                    - Specific: "4 người" → num_of_people = ["4"]
                  + If NOT mentioned → KEEP as []

                - **calories**: IMPORTANT (soft_constraints)
                  + Update if user mentions calories (including "don't care")
                    - "không cần quan tâm kcal" → calories = ["none"]
                    - Specific: "1000 kcal" → calories = ["1000 kcal"]
                  + If NOT mentioned → KEEP as []

                **Example (Vietnamese):**

                Input: "Gợi ý món Tết, nguyên liệu nem chua, không yêu cầu thời gian nấu, tôi không bị dị ứng, không cần quan tâm số người ăn và số kcal"

                Analysis:
                - Mentions "món Tết" → Update type_of_food
                - Mentions "nem chua" → Update ingredients  
                - Mentions "không yêu cầu thời gian" → Update cook_time (all 3 types)
                - Mentions "không bị dị ứng" → Update algeric = none
                - Mentions "không cần quan tâm số người" → Update num_of_people = none
                - Mentions "không cần quan tâm kcal" → Update calories = none

                → ALL 6 fields should be updated!

                IMPORTANT: Return ONLY pure JSON, NO markdown, NO ```json, NO explanation.
                Return valid JSON with all commas and brackets in correct positions.
                """
        
        resp = genai.GenerativeModel(MODEL).generate_content(prompt)
        
        try:
            # Parse LLM response
            text = resp.text.strip()
            if "```" in text:
                parts = text.split("```")
                if len(parts) >= 2:
                    text = parts[1]
                    if text.startswith("json") or text.startswith("JSON"):
                        text = text[4:]
                    text = text.strip()
            text = text.strip()
            
            new_state = json.loads(text)
            
            # Validate structure
            if "hard_constraints" in new_state and "soft_constraints" in new_state:
                state = new_state
            else:
                print("⚠️ LLM returned incomplete JSON structure, keeping old state")
                print(f"Response: {text[:200]}...")
                
        except json.JSONDecodeError as e:
            print(f"❌ Error parsing JSON: {e}")
            print(f"Response: {resp.text[:300]}...")
            print("⚠️ Keeping old state, continuing...")
        except Exception as e:
            print(f"❌ Unexpected error: {e}")
            print(f"Response: {resp.text[:300]}...")
            print("⚠️ Keeping old state, continuing...")
    
    # Handle "Accept Recommendation" intent
    if "Accept Recommendation" in intents:
        # Extract dish name from user utterance using LLM
        extract_prompt = f"""
                        User says: "{user_utterance}"

                        Extract the DISH NAME that the user is accepting/liking.
                        Return ONLY the dish name, NO explanation.
                        If no dish name found, return "NONE".

                        Examples (Vietnamese):
                        - "Tôi thích món Nem rán" → "Nem rán"
                        - "Món bánh xèo này ok" → "Bánh xèo"
                        - "Cho tôi thêm phở bò" → "Phở bò"
                        """
        dish_name = genai.GenerativeModel(MODEL).generate_content(extract_prompt).text.strip()
        
        if dish_name and dish_name != "NONE":
            if dish_name not in state["accepted_items"]:
                state["accepted_items"].append(dish_name)
                print(f"Added '{dish_name}' to accepted items")
    
    # Handle "Reject Recommendation" intent
    if "Reject Recommendation" in intents:
        # Extract dish name from user utterance using LLM
        extract_prompt = f"""
                        User says: "{user_utterance}"

                        Extract the DISH NAME that the user is rejecting/disliking.
                        Return ONLY the dish name, NO explanation.
                        If no dish name found, return "NONE".

                        Examples (Vietnamese):
                        - "Tôi không thích món Nem rán" → "Nem rán"
                        - "Món bánh xèo không hợp" → "Bánh xèo"
                        - "Bỏ phở bò đi" → "Phở bò"
                        """
        dish_name = genai.GenerativeModel(MODEL).generate_content(extract_prompt).text.strip()
        
        if dish_name and dish_name != "NONE":
            if dish_name not in state["rejected_items"]:
                state["rejected_items"].append(dish_name)
                print(f"Added '{dish_name}' to rejected items")
    
    return state

### STEP 3: ACTION SELECTION

In [ ]:
# STEP 3: ACTION SELECTION

def select_action(intents, state):
    # If user is asking a question
    if "Inquire" in intents:
        return "Answer"
    
    # Check if all hard constraints are filled
    for key in state["hard_constraints"]:
        if len(state["hard_constraints"][key]) == 0:
            return "Request Information"
    
    # Check if soft constraints are filled
    all_soft_empty = all(len(state["soft_constraints"][key]) == 0 
                        for key in state["soft_constraints"])
    
    if all_soft_empty:
        return "Request Information"
    
    # All information collected
    return "Info Complete"

### STEP 4: RESPONSE GENERATION

In [ ]:
# STEP 4: RESPONSE GENERATION - WITH RECIPES BUFFER

def answer_question_with_buffer(user_utterance, state):
    # Get recipes from buffer
    recipes_context = get_recipes_buffer_text()
    
    # Load current state
    state_context = json.dumps(state, indent=2, ensure_ascii=False)
    
    answer_prompt = f"""
Bạn là trợ lý gợi ý món ăn. Trả lời câu hỏi của user dựa trên:

1. DANH SÁCH MÓN ĐÃ GỢI Ý (có đầy đủ thông tin):
{recipes_context}

2. YÊU CẦU CỦA USER (state.json):
{state_context}

3. USER HỎI: "{user_utterance}"

NHIỆM VỤ - PHÂN TÍCH CÂU HỎI:
- Nếu hỏi về "cách làm", "các bước", "làm thế nào", "nấu như thế nào" → Đưa ra "Các bước nấu"
- Nếu hỏi về "nguyên liệu", "cần gì" → Đưa ra "Nguyên liệu"
- Nếu hỏi về "thời gian" → Đưa ra "Thời gian"
- Nếu hỏi về "lưu ý", "mẹo", "tips" → Đưa ra "Lưu ý"
- Nếu hỏi tổng quát về món → Tóm tắt thông tin chính

YÊU CẦU:
- Trả lời bằng tiếng Việt, tự nhiên và thân thiện
- KHÔNG dùng kiến thức nội tại, CHỈ dựa vào data trong danh sách
- Nếu user hỏi về cách làm/bước nấu → PHẢI đưa ra các bước từ trường "Các bước nấu"
- Trả lời đầy đủ, chi tiết khi cần
- Nếu có link, đưa link cho user để xem thêm
- Nếu không có thông tin → Nói "Tôi chưa có thông tin này trong danh sách món đã gợi ý"
    """
    response = genai.GenerativeModel(MODEL).generate_content(answer_prompt)
    return response.text


def generate_response(user_utterance, action, state):
    if action == "Request Information":
        # Find missing constraint and ask for it
        
        # Check hard constraints first
        for key in state["hard_constraints"]:
            if len(state["hard_constraints"][key]) == 0:
                questions = {
                    "type_of_food": "Bạn muốn tìm loại món gì? (ví dụ: Món kho, món luộc, món xào...)",
                    "ingredients": "Bạn muốn món có nguyên liệu gì? (ví dụ: thịt lợn, hải sản, rau...)",
                }
                return questions.get(key, f"Could you provide information about {key}?")
        
        # Then check soft constraints
        for key in state["soft_constraints"]:
            if len(state["soft_constraints"][key]) == 0:
                questions = {
                    "cook_time": "Bạn muốn món nấu trong bao lâu?",
                    "num_of_people": "Bạn muốn món ăn nấu cho bao nhiêu người?",
                    "calories": "Bạn quan tâm đến mức calories không?",
                    "algeric": "Bạn có dị ứng với thành phần nào không? (nếu không thì nói 'không')"
                }
                return questions.get(key, f"Could you provide information about {key}?")
        
        return "Tôi cần thêm thông tin để giúp bạn."

    elif action == "Answer":  
        # Use recipes buffer + state to answer    
        return answer_question_with_buffer(user_utterance, state)                   

    elif action == "Info Complete":
        return "Tôi đã có đầy đủ thông tin cần thiết, bạn hãy đợi tôi 1 chút nhé"

## 4. Import utilities from 5_Late_fusion

### 4.1 Search recipes_late_fusion

In [ ]:
def search_recipes_late_fusion(query, model, recipes_embeddings_list, all_recipes_df, top_k=10):
    """
    Search recipes using LATE FUSION strategy (Average Similarity)

    Late Fusion = Tính similarity với TẤT CẢ câu trong món → Average → Rank

    Args:
        query: User's search query (Vietnamese)
        model: SentenceTransformer model
        recipes_embeddings_list: List of embeddings per dish
        all_recipes_df: Recipe metadata dataframe
        top_k: Number of results to return

    Returns:
        DataFrame with top_k recipes and similarity scores
    """
    # 1. Encode query
    query_embedding = model.encode([query])
    query_embedding = query_embedding / np.linalg.norm(query_embedding)  # Normalize

    # 2. Calculate average similarity for EACH recipe
    recipe_scores = []

    for recipe_idx, dish_embeds in enumerate(recipes_embeddings_list):
        if len(dish_embeds) == 0:
            continue

        # Normalize dish embeddings
        dish_embeds_norm = dish_embeds / np.linalg.norm(dish_embeds, axis=1, keepdims=True)

        # Compute cosine similarity với TẤT CẢ câu
        similarities = np.dot(dish_embeds_norm, query_embedding.T).flatten()

        # LATE FUSION: Average similarity
        avg_similarity = np.mean(similarities)

        recipe_scores.append({
            'recipe_idx': recipe_idx,
            'avg_similarity': float(avg_similarity),
            'max_similarity': float(np.max(similarities)),
            'min_similarity': float(np.min(similarities)),
            'num_sentences': len(similarities)
        })

    # 3. Sort by average similarity
    recipe_scores.sort(key=lambda x: x['avg_similarity'], reverse=True)
    top_recipes = recipe_scores[:top_k]

    # 4. Create results dataframe with FULL recipe info
    results = []
    for item in top_recipes:
        recipe_idx = item['recipe_idx']
        recipe = all_recipes_df.iloc[recipe_idx]

        results.append({
            'recipe_idx': recipe_idx,
            'avg_similarity': item['avg_similarity'],
            'max_similarity': item['max_similarity'],
            'min_similarity': item['min_similarity'],
            'num_sentences': item['num_sentences'],
            'title': recipe['title'],
            'type_of_food': recipe['type_of_food'],
            'cook_time': recipe['cook_time'],
            'num_of_people': recipe['num_of_people'],
            'ingredients': recipe['ingredients'],
            'step': recipe['step'],
            'note': recipe['note'],
            'description': recipe['description'],
            'link': recipe['link']  # Thêm link
        })

    return pd.DataFrame(results)

### 4.2 Display result function

In [ ]:
def generate_query_from_state(state):
    """
    Generate search query from state.json constraints using rule-based approach
    
    Args:
        state: Dictionary from state.json (with hard_constraints, soft_constraints)
    
    Returns:
        Generated query string
    """
    parts = []
    
    # Hard constraints (priority)
    if "hard_constraints" in state:
        # Type of food
        if state["hard_constraints"].get("type_of_food"):
            type_food = state["hard_constraints"]["type_of_food"]
            if type_food and type_food[0] != "none":
                parts.append(type_food[0])
        
        # Ingredients
        if state["hard_constraints"].get("ingredients"):
            ingredients = state["hard_constraints"]["ingredients"]
            if ingredients and ingredients != ["none"]:
                if len(ingredients) == 1:
                    parts.append(f"có {ingredients[0]}")
                else:
                    parts.append(f"có {', '.join(ingredients)}")
    
    # Soft constraints
    if "soft_constraints" in state:
        # Number of people
        if state["soft_constraints"].get("num_of_people"):
            num_people = state["soft_constraints"]["num_of_people"]
            if num_people and num_people[0] != "none":
                parts.append(f"cho {num_people[0]} người")
        
        # Cook time
        if state["soft_constraints"].get("cook_time"):
            cook_time = state["soft_constraints"]["cook_time"]
            if cook_time and cook_time[0] != "none":
                parts.append(f",có thời gian nấu {cook_time[0]} phút")
    
    # Combine parts into natural query
    if not parts:
        return "Món ăn"
    
    query = " ".join(parts)
    return query


## 5. Gemini Presentation Function (Format RecSys results for user)

In [ ]:
def load_state_json(filepath=STATE_FILE):
    """Load dialogue state from JSON file"""
    try:
        if Path(filepath).exists():
            with open(filepath, 'r', encoding='utf-8') as f:
                state = json.load(f)
            return state
        else:
            print(f"[Error]: {filepath} not found")
            return None
    except json.JSONDecodeError:
        print(f"[Error]: Invalid JSON in {filepath}")
        return None

## 5. Gemini Presentation Function (Format RecSys results for user)

In [ ]:
# GEMINI PRESENTATION WITH RECIPES BUFFER

def present_recommendations_with_gemini(results_df, state):
    """
    Present recommendations + save to recipes buffer for future reference
    """
    # Format full recipe data
    recipes_data = []
    for idx, row in results_df.iterrows():
        recipe = {
            "index": idx + 1,
            "title": row['title'],
            "type_of_food": row['type_of_food'],
            "cook_time": row['cook_time'],
            "num_of_people": row['num_of_people'],
            "similarity_score": float(row['avg_similarity']),
            "ingredients": row['ingredients'] if pd.notna(row['ingredients']) else 'N/A',
            "description": row['description'] if pd.notna(row['description']) else 'N/A',
            "step": row['step'] if pd.notna(row['step']) else 'N/A',
            "note": row['note'] if pd.notna(row['note']) else 'N/A',
            "link": row['link'] if pd.notna(row['link']) else 'N/A'
        }
        recipes_data.append(recipe)
    
    prompt = f"""
Bạn là trợ lý gợi ý món ăn thông minh. Bạn nhận được:

1. YÊU CẦU CỦA NGƯỜI DÙNG (state.json):
{json.dumps(state, indent=2, ensure_ascii=False)}

2. KẾT QUẢ TÌM KIẾM TỪ HỆ THỐNG (Top {len(recipes_data)} món):
{json.dumps(recipes_data, indent=2, ensure_ascii=False)}

NHIỆM VỤ CỦA BẠN:

1. **QUAN TRỌNG - LỌC DỊ ỨNG:**
   - Kiểm tra trường "algeric" trong soft_constraints
   - Nếu có dị ứng (không phải "none"), loại BỎ các món có thành phần dị ứng trong "ingredients"
   - Chỉ giới thiệu các món an toàn

2. **GỢI Ý THÔNG MINH:**
   - Phân tích món nào phù hợp nhất với yêu cầu (thời gian, số người, loại món)
   - Đề xuất 3-5 món nổi bật với lý do cụ thể
   - Sử dụng tone thân thiện: "Tôi nghĩ món A sẽ ... Hoặc bạn cũng có thể thử món B vì..."

3. **FORMAT TRẢ LỜI:**
   - Giới thiệu ngắn gọn (2-3 câu)
   - Gợi ý 3-5 món với:
     + Tên món (in đậm)
     + Lý do phù hợp (ngắn gọn)
     + Thông tin quan trọng (thời gian, nguyên liệu chính)
     + Link để xem chi tiết
   - Kết thúc: "Bạn muốn biết thêm chi tiết món nào không?"

YÊU CẦU:
- Viết bằng tiếng Việt tự nhiên, thân thiện
- Giải thích CỤ THỂ tại sao món này phù hợp
- Nếu có món bị loại do dị ứng, KHÔNG đề cập
- Đưa đầy đủ thông tin để user có thể hỏi thêm sau này
    """
    
    response = genai.GenerativeModel(MODEL).generate_content(prompt)
    gemini_response = response.text
    
    # Update recommended_items in state
    recommended_titles = [recipe["title"] for recipe in recipes_data[:5]]
    state["recommended_items"] = recommended_titles
    save_state(state)
    
    # Save to recipes buffer (langchain-style memory)
    add_recipes_to_buffer(recipes_data[:5])  # Keep top 5 for buffer
    
    return gemini_response

## FULL CONVERSATION LOOP

In [ ]:
def check_restart_intent(user_utterance):
    """Check if user wants to restart the conversation"""
    restart_keywords = [
        "bắt đầu lại", "bat dau lai", "restart", "reset",
        "làm lại", "lam lai", "start over", "bắt đầu từ đầu"
    ]
    return any(keyword in user_utterance.lower() for keyword in restart_keywords)


def reset_conversation():
    """Reset state and buffer to start fresh"""
    global RECOMMENDED_RECIPES_BUFFER
    RECOMMENDED_RECIPES_BUFFER = []
    with open(STATE_FILE, "w", encoding="utf-8") as f:
        json.dump(DEFAULT_STATE, f, indent=4, ensure_ascii=False)
    print("\nĐã reset hệ thống. Bắt đầu lại từ đầu!\n")


print("=" * 80)
print("RA-Rec Food Recommender System - FULL CONVERSATION LOOP")
print("=" * 80)
print()

# Initial reset
reset_conversation()

# Greeting
greeting = "Xin chào! Tôi là trợ lý gợi ý món ăn.\nBạn muốn tìm món ăn gì? (ví dụ: món Tết, món nhanh, món cho 4 người...)\nType 'exit' hoặc 'restart' để thoát/bắt đầu lại."
print(f"BOT: {greeting}\n")

# Main conversation loop
while True:
    # Get user input
    user_utterance = input("USER: ").strip()
    
    # Check for exit
    if user_utterance.lower() in ["exit", "stop", "quit"]:
        print("\nBOT: Cảm ơn bạn đã sử dụng hệ thống. Hẹn gặp lại! 👋")
        break
    
    # Check for restart
    if check_restart_intent(user_utterance):
        reset_conversation()
        print(f"BOT: {greeting}\n")
        continue
    
    if not user_utterance:
        continue
    
    # Load current state
    state = load_state()
    
    # STEP 1: Intent Classification
    intents = classify_intent(user_utterance)
    
    # Special handling: Force "Provide Preference" if we're in info-gathering mode
    asking_hard = any(len(state["hard_constraints"][key]) == 0 
                     for key in state["hard_constraints"])
    if asking_hard and "Provide Preference" not in intents:
        intents = ["Provide Preference"]
    
    # Special handling for "không" (no) responses
    if any(word in user_utterance.lower() 
           for word in ["không", "khong", "không có", "không dị ứng"]):
        if "algeric" in state["soft_constraints"] and state["soft_constraints"]["algeric"] == []:
            state["soft_constraints"]["algeric"] = ["none"]
            save_state(state)
            if "Provide Preference" not in intents:
                intents.append("Provide Preference")
    
    # Handle rejection of soft constraints
    if any(word in user_utterance.lower() 
           for word in ["không", "khong", "no", "skip", "nope", "don't", "dont", "bỏ qua"]):
        all_hard_filled = all(len(state["hard_constraints"][key]) > 0 
                             for key in state["hard_constraints"])
        if all_hard_filled:
            for key in state["soft_constraints"]:
                if len(state["soft_constraints"][key]) == 0:
                    state["soft_constraints"][key] = ["none"]
            save_state(state)
            
            action = select_action(intents, state)
            reply = generate_response(user_utterance, action, state)
            print(f"BOT: {reply}\n")
            
            if action == "Info Complete":
                break
            continue
    
    # STEP 2: State Update
    state = update_state(user_utterance, intents, state)
    save_state(state)
    
    # STEP 3: Action Selection
    action = select_action(intents, state)
    
    # STEP 4: Response Generation
    reply = generate_response(user_utterance, action, state)
    print(f"BOT: {reply}\n")
    
    # Check if dialogue is complete
    if action == "Info Complete":
        print("=" * 80)
        print("Đã thu thập đủ thông tin! Bắt đầu tìm kiếm món ăn...")
        print("=" * 80)
        break

print("\n" + "=" * 80)
print("PHASE 2: RECOMMENDER SYSTEM SEARCH")
print("=" * 80)

# Generate query from state
state = load_state()
query = generate_query_from_state(state)
print(f"Query: {query}\n")

# Run Late Fusion search
results = search_recipes_late_fusion(
    query=query,
    model=model,
    recipes_embeddings_list=recipes_embeddings_list,
    all_recipes_df=all_recipes_df,
    top_k=10
)

print("=" * 80)
print("PHASE 3: GEMINI PRESENTATION")
print("=" * 80)

# Present results with Gemini (saves to buffer automatically)
gemini_response = present_recommendations_with_gemini(results, state)

print("BOT:")
print(gemini_response)

# PHASE 4: Interactive feedback loop
print("\n" + "=" * 80)
print("PHASE 4: FEEDBACK & REFINEMENT")
print("=" * 80)
feedback_msg = "Bạn có thể hỏi thêm về món nào, hoặc nói 'tôi thích món X' / 'cho tôi món khác'.\nGõ 'done' để kết thúc, 'quit' để thoát hẳn."
print(f"BOT: {feedback_msg}\n")

SHOULD_EXIT = False  # Flag để track exit command

while True:
    user_feedback = input("USER: ").strip()
    
    # Check for done
    if user_feedback.lower() in ["done", "xong"]:
        goodbye = "Cảm ơn bạn! Chúc bạn nấu ăn ngon miệng!"
        print(f"\nBOT: {goodbye}")
        break
    
    # Check for exit - set flag để thoát khỏi cả outer loop
    if user_feedback.lower() in ["exit", "stop", "quit", "thoát"]:
        print("\nBOT: Cảm ơn bạn đã sử dụng hệ thống. Hẹn gặp lại!")
        SHOULD_EXIT = True
        break
    
    # Check for restart
    if check_restart_intent(user_feedback):
        reset_conversation()
        print(f"BOT: {greeting}")
        print("\nBạn sẽ được đưa về đầu cuộc trò chuyện...")
        # Break to outer loop to restart
        break
    
    if not user_feedback:
        continue
    
    # Classify feedback intent
    state = load_state()
    intents = classify_intent(user_feedback)
    
    # Update accepted/rejected items
    state = update_state(user_feedback, intents, state)
    save_state(state)
    
    # Generate response based on intent
    if "Accept Recommendation" in intents:
        response = f"Tuyệt vời! Tôi sẽ lưu lại sở thích này của bạn. \nĐã thích: {', '.join(state['accepted_items'])}"
        print(f"BOT: {response}")
        
    elif "Reject Recommendation" in intents:
        response = "Được rồi! Để tôi tìm món khác phù hợp hơn..."
        print(f"BOT: {response}")
        
        # Filter out rejected items
        filtered_results = results[~results['title'].isin(state['rejected_items'])]
        
        if len(filtered_results) > 0:
            new_response = present_recommendations_with_gemini(filtered_results.head(5), state)
            print(f"\n{new_response}")
        else:
            no_more = "Xin lỗi, không còn món nào khác phù hợp. Bạn có thể nói 'restart' để tìm món hoàn toàn mới."
            print(f"BOT: {no_more}")
    
    elif "Inquire" in intents:
        # Answer using recipes buffer + state
        answer = answer_question_with_buffer(user_feedback, state)
        print(f"BOT: {answer}")
    
    else:
        # General response
        general = "Tôi có thể giúp gì thêm cho bạn?"
        print(f"BOT: {general}")
    
    print()

# Check if user wants to exit completely
if SHOULD_EXIT:
    # Exit the program completely
    import sys
    sys.exit(0)